In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [75]:
ad_df = pd.read_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/Conversion-Lens/data/ab_data.csv")
ad_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294480 entries, 0 to 294479
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294480 non-null  int64 
 1   timestamp     294480 non-null  object
 2   group         294480 non-null  object
 3   landing_page  294480 non-null  object
 4   converted     294480 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [76]:
ad_df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


### Info
- No Null values
- datatype problme with timedelta 

NOTE:- timestamp is the time spend by user on the ecommerce website [for both converted and non converted scenerio]

In [77]:
## data type conversion 
print(ad_df["timestamp"].sample(10)) # format = mm:ss.s
print(ad_df["timestamp"].shape[0])


184859    24:29.2
163991    01:22.7
36463     10:18.4
25557     59:04.4
78491     03:57.5
148657    34:04.6
148282    35:06.1
149409    13:27.2
172250    45:40.3
213393    38:52.7
Name: timestamp, dtype: object
294480


In [78]:
ad_df["timestamp"] = pd.to_timedelta('00:' + ad_df["timestamp"])
print(ad_df["timestamp"].sample(10)) # format = mm:ss.s
print(ad_df["timestamp"].shape[0])

244890   0 days 00:24:14.100000
264024   0 days 00:28:42.200000
71214    0 days 00:29:53.800000
46869    0 days 00:19:08.800000
87921    0 days 00:05:44.500000
130158   0 days 00:44:05.500000
62476    0 days 00:42:53.400000
64067    0 days 00:03:34.900000
257996   0 days 00:07:37.500000
245528   0 days 00:33:37.900000
Name: timestamp, dtype: timedelta64[ns]
294480


In [79]:
ad_df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,0 days 00:11:48.600000,control,old_page,0
1,804228,0 days 00:01:45.200000,control,old_page,0
2,661590,0 days 00:55:06.200000,treatment,new_page,0
3,853541,0 days 00:28:03.100000,treatment,new_page,0
4,864975,0 days 00:52:26.200000,control,old_page,1


### Data Exploration

In [80]:
# Unique users
d1 = ad_df["user_id"].value_counts().reset_index(name="count").sort_values(by="count", ascending = False)
is1 = d1.iloc[0:4]
print(ad_df[ad_df["user_id"].isin(is1.user_id)].sort_values(by="user_id"))

        user_id              timestamp      group landing_page  converted
47219    654395 0 days 00:12:25.400000  treatment     new_page          1
179175   654395        0 days 00:57:21    control     new_page          0
56183    703112 0 days 00:07:53.400000    control     new_page          1
113672   703112 0 days 00:19:42.100000  treatment     new_page          0
110111   712134 0 days 00:37:21.200000  treatment     old_page          0
129192   712134 0 days 00:51:38.800000    control     old_page          0
221114   856917 0 days 00:17:16.200000    control     new_page          0
240173   856917 0 days 00:01:20.800000  treatment     new_page          0


In [81]:
print(d1[d1["count"] == 2].shape[0])

3895


### data integrity issue

Treatment group should be shown new landing page and Control group should be shown old_page, this might be tracking script error, or session cookies exprires. Dropping those user entirely from the dataset.

In [82]:
ad_df = ad_df[((ad_df['group'] == 'treatment') & (ad_df['landing_page'] == 'new_page')) |
              ((ad_df['group'] == 'control') & (ad_df['landing_page'] == 'old_page'))]

# Check the new shape (should be 290,585 rows)
print(ad_df.shape[0])

290587


In [83]:
d1 = ad_df["user_id"].value_counts().reset_index(name="count").sort_values(by="count", ascending = False)
print(d1.head())

        user_id  count
0        759899      2
1        773192      2
290583   889019      1
193720   803683      1
193726   679687      1


In [88]:
ad_df[ad_df["user_id"] == 759899]
# this is correct as it is two instances by the user 759899 from the treatment group.

,user_id,timestamp,group,landing_page,converted
250001,759899,0 days 00:07:36.100000,treatment,new_page,0
294478,759899,0 days 00:20:29,treatment,new_page,0


In [93]:
print(ad_df.converted.value_counts())
print(f"Total conversion rate is {ad_df.converted.mean() * 100.0}")

converted
0    255834
1     34753
Name: count, dtype: int64
Total conversion rate is 11.959585253297636
